# សម័យទី 2 – បណ្ដាញ RAG ខ្នាតតូច

បង្កើតបណ្ដាញ Retrieval-Augmented Generation ខ្នាតស្រាល ដោយប្រើ Foundry Local + sentence-transformers embeddings.


### ពន្យល់៖ ការដំឡើងអាស្រ័យភាព
ដំឡើងកញ្ចប់អប្បបរិមាណសម្រាប់បណ្តាញដំណើរការ​នេះ:
- `foundry-local-sdk` សម្រាប់ការគ្រប់គ្រងម៉ូឌែលក្នុងស្រុក (បើមិនកំពុងប្រើផ្លូវ BASE_URL ផ្ទាល់).
- `openai` សម្រាប់រចនាសម្ព័ន្ធ SDK ដែលឆ្គងគ្នា (មានឧបករណ៍ជួយខ្លះៗ).
- `sentence-transformers` សម្រាប់ embeddings.
- `numpy` សម្រាប់គណិតវិទ្យាវ៉ិចទ័រ.
អាចរត់ឡើងវិញបានដោយសុវត្ថិភាព; អាចរំលងប្រសិនបើបរិយាកាសបានបំពេញរួច។


# សេណារីយ៉ូ
កំណត់ត្រានេះបង្កើតបន្ទាត់ដំណើរការតិចតួចសម្រាប់ Retrieval-Augmented Generation (RAG) ដែលដំណើរការទាំងមូលក្នុងស្រុក:
- ភ្ជាប់ទៅកាន់ម៉ូដែល Foundry Local (ស្វ័យឆ្លាតស្គាល់តាម SDK ឬ BASE_URL).
- បង្កើតបណ្ណាល័យឯកសារតូចមួយនៅក្នុងចងចាំ និងបញ្ចូលវាដោយ Sentence Transformers.
- អនុវត្តការស្វែងរកស្រដៀងវ៉ិចទ័រយ៉ាងសាមញ្ញ (គ្មានសន្ទស្សន៍ខាងក្រៅ) សម្រាប់ភាពច្បាស់លាស់.
- បញ្ចូនសំណើបង្កើតដែលមានមូលដ្ឋានតាមរយៈច្រក fallback ជាច្រើនរបស់ HTTP (`/v1/chat/completions`, `/v1/completions`, `/v1/responses`).
- ផ្តល់ជំនួយការ `answer()` ដែលព្យាយាមឡើងវិញជាមួយទម្រង์ម៉ូដែលជំនួសនៅពេលការព្យាយាមដំបូងបរាជ័យ.

ប្រើនេះជាទម្រង់រៀបចំសម្រាប់រកមូលហេតុនិងពិសោធមុនពេលពង្រីកទៅកាន់បណ្ណាល័យធំៗ ស្តុកវ៉ិចទ័រត្រូវបានរក្សាទុក ឬម៉េត្រិចវាយតម្លៃ (សូមមើល RAG evaluation notebook).


In [5]:
# Install dependencies
!pip install -q foundry-local-sdk openai sentence-transformers numpy

### ការពន្យល់៖ ការនាំចូលមូលដ្ឋាន

ផ្ទុកបណ្ណាល័យមូលដ្ឋានដែលចាំបាច់សម្រាប់ការបង្កើត embedding និងការប៉ាន់ស្មានក្នុងស្រុក:
- SentenceTransformer សម្រាប់ការបង្កើត embedding វិចទ័រដង់ស។
- FoundryLocalManager (ជម្រើស) ដើម្បីគ្រប់គ្រងសេវាកម្មក្នុងស្រុក។
- OpenAI client សម្រាប់រូបរាងវត្ថុដែលស្គាល់ (បើទោះបីជា យើងបន្ទាប់មកហៅ HTTP ដោយផ្ទាល់ក៏ដោយ).


In [6]:
import os, numpy as np
from sentence_transformers import SentenceTransformer
from foundry_local import FoundryLocalManager
from openai import OpenAI

### ការពន្យល់: សំណុំឯកសារគំរូ
កំណត់បញ្ជីតូចមួយនៅក្នុងចងចាំសម្រាប់ប្រកាសទាក់ទងនឹងដែន។ ធ្វើឱ្យការធ្វើម្តងៗលឿន និងគ្រប់គ្រងបាន ដើម្បីឲ្យការផ្ដោតនៅលើយន្តការនៃបណ្តាញដំណើរការ (ការទាញយក + ការផ្ដល់មូលដ្ឋាន) កើតមាន ជាងការរៀបចំទិន្នន័យ។


In [7]:
DOCS = [
    'Foundry Local provides an OpenAI-compatible local inference endpoint.',
    'Retrieval Augmented Generation improves answer grounding by injecting relevant context.',
    'Edge AI reduces latency and preserves privacy via local execution.',
    'Small Language Models can offer competitive quality with lower resource usage.',
    'Vector similarity search retrieves semantically relevant documents.'
]

### ពន្យល់: ការតភ្ជាប់, ការជ្រើសម៉ូដែល & ការចាប់ផ្តើម Embedding
យុទ្ធសាស្ត្រតភ្ជាប់ដែលរឹងមាំ:
1. ជាជម្រើស ប្រើ `BASE_URL` ដោយច្បាស់ (ផ្លូវ HTTP បរិសុទ្ធ) បើមិនមាន វានឹងត្រឡប់ទៅប្រើ FoundryLocalManager.
2. ធ្វើសាក `/v1/models` ហើយជ្រើស id ម៉ូដែលជាក់លាក់ដែលសមស្របបំផុត (alias ត្រឹមត្រូវ > គ្រួសារផ្លូវការ > ដែលមានស្រាប់ជាលើកដំបូង).
3. អនុវត្តរង្វិលព្យាយាមឡើងវិញជាមួយកំណត់តម្លៃ `FOUNDRY_CONNECT_RETRIES` & ការពន្យារពេល.
4. ចាប់ផ្តើម embeddings របស់ SentenceTransformer (វ៉ិចទ័រដែលបានធម្មតា) សម្រាប់ក្រុមអត្ថបទគំរូ.
5. កត់ត្រាកំណែ OpenAI SDK សម្រាប់ភាពអាចបង្កើតឡើងវិញ.
បើសេវាកម្មអវត្តមាន វានឹងបង្ហាញការណែនាំដើម្បីចាប់ផ្តើមវា ជំនួសការរលំ.


In [12]:
import os, time, json, requests, re
# Native Foundry Local SDK preferred; fall back to explicit BASE_URL if provided
os.environ.setdefault('FOUNDRY_LOCAL_ALIAS', 'phi-4-mini')
alias = os.getenv('FOUNDRY_LOCAL_ALIAS', os.getenv('TARGET_MODEL', 'phi-4-mini'))
base_url_env = os.getenv('BASE_URL', '').strip()
manager = None
client = None
endpoint = None

def _canonicalize(model_id: str) -> str:
    """Remove CUDA suffix and version tags from model name."""
    b = model_id.split(':')[0]
    return re.sub(r'-cuda.*', '', b)

try:
    if base_url_env:
        # Allow user override; normalize by removing trailing / and optional /v1
        root = base_url_env.rstrip('/')
        if root.endswith('/v1'):
            root = root[:-3]
        endpoint = root
        print(f'[INFO] Using explicit BASE_URL override: {endpoint}')
    else:
        from foundry_local import FoundryLocalManager
        manager = FoundryLocalManager(alias)
        # Manager endpoint already includes /v1 - remove it for our base
        raw_endpoint = manager.endpoint.rstrip('/')
        if raw_endpoint.endswith('/v1'):
            endpoint = raw_endpoint[:-3]
        else:
            endpoint = raw_endpoint
        print(f'[OK] Foundry Local manager endpoint: {manager.endpoint} | base={endpoint} | alias={alias}')
    
    # Probe models list (endpoint does NOT include /v1 here)
    models_resp = requests.get(endpoint + '/v1/models', timeout=5)
    models_resp.raise_for_status()
    payload = models_resp.json() if models_resp.headers.get('content-type','').startswith('application/json') else {}
    data = payload.get('data', []) if isinstance(payload, dict) else []
    ids = [m.get('id') for m in data if isinstance(m, dict)]
    
    # Select best matching model
    chosen = None
    if alias in ids:
        chosen = alias
    else:
        for mid in ids:
            if _canonicalize(mid) == _canonicalize(alias):
                chosen = mid
                break
    if not chosen and ids:
        chosen = ids[0]
    model_name = chosen or alias
    
    # Initialize OpenAI client
    from openai import OpenAI as _OpenAI
    client = _OpenAI(
        base_url=endpoint + '/v1',  # OpenAI client needs full base URL with /v1
        api_key=(getattr(manager, 'api_key', None) or os.getenv('API_KEY') or 'not-needed')
    )
    print(f'[OK] Model resolved: {model_name} (total_models={len(ids)})')
except Exception as e:
    print('[ERROR] Failed to initialize Foundry Local client:', e)
    client = None
    model_name = alias

# Expose BASE for downstream compatibility (without /v1)
BASE = endpoint

# Embeddings setup
embed_model_name = os.getenv('EMBED_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')
try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(embed_model_name)
    doc_emb = embedder.encode(DOCS, convert_to_numpy=True, normalize_embeddings=True)
    print(f'[OK] Embedded {len(DOCS)} docs using {embed_model_name} shape={doc_emb.shape}')
except Exception as e:
    print('[ERROR] Embedding init failed:', e)
    embedder = None
    doc_emb = None

try:
    import openai as _openai
    openai_version = getattr(_openai, '__version__', 'unknown')
    print('OpenAI SDK version:', openai_version)
except Exception:
    openai_version = 'unknown'

if client is None:
    print('\nNEXT: Start/verify service then re-run this cell:')
    print('  foundry service start')
    print('  foundry model run phi-4-mini')
    print('  (optional) set BASE_URL=http://127.0.0.1:57127')

[OK] Foundry Local manager endpoint: http://127.0.0.1:59778/v1 | base=http://127.0.0.1:59778 | alias=phi-4-mini
[OK] Model resolved: deepseek-r1-distill-qwen-7b-cuda-gpu:0 (total_models=11)
[OK] Embedded 5 docs using sentence-transformers/all-MiniLM-L6-v2 shape=(5, 384)
OpenAI SDK version: 1.109.1


### ការពិពណ៌នា: មុខងារ Retrieve (ស្រដៀងភាពវ៉ិចទ័រ)
`retrieve(query, k=3)` បំលែងសំណួរ, គណនាពីភាពស្រដៀងកូស៊ីន (dot product លើវ៉ិចទ័រដែលបានធម្មតា), ហើយត្រឡប់សន្ទស្សន៍ឯកសារ top-k។ នេះនៅតែសាមញ្ញ & ស្ថិតក្នុងចងចាំ សម្រាប់ភាពច្បាស់លាស់។


In [9]:
def retrieve(query, k=3):
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = doc_emb @ q
    return sims.argsort()[::-1][:k]

### ការពន្យល់៖ ការបង្កើតផ្អែកលើ SDK និងឧបករណ៍ជំនួយចម្លើយ
បានធ្វើប្ដូរឡើងដើម្បីប្រើ Foundry Local SDK និងវិធីសាស្រ្ត client ដែលសមស្របជាមួយ OpenAI ជំនួសការប៉ុស៍ HTTP ដោយដៃ:
- ផ្លូវចម្បង: `client.chat.completions.create` (សារដែលមានរចនាសម្ព័ន្ធ).
- ជំនួស: `client.completions.create` (prompt បុរាណ) បន្ទាប់មក `client.responses.create` (API ចម្លើយដែលបានសម្រួល).
- ធ្វើឲ្យស្ដង់ដារនៃ id ម៉ូឌែលជាជម្រើស (RAW ទល់នឹង ALT ដែលបានកាត់) ដើម្បីពង្រីកភាពសមស្រប។
- `answer()` បង្កើត prompt ដែលមានមូលដ្ឋានពីឯកសារ top-k ដែលយកបាន ហើយកត់ត្រាកំណត់ត្រាការព្យាយាមតាមលំដាប់.
នេះធ្វើឲ្យលក្ខខណ្ឌនៃយុទ្ធសាស្រ្តអាចអានបាន ខណៈពេលដែលវាក៏ផ្តល់នូវការធ្លាក់ចុះយ៉ាងទន់ភ្លន់សម្រាប់ចំណុចបញ្ចប់ដែលកំពុងអភិវឌ្ឍន៍ និងសមស្របជាមួយ OpenAI។


In [14]:
# SDK-based generation (Foundry Local manager + OpenAI client methods)
import re, time, json

def _strip_model_name(name: str) -> str:
    """Strip CUDA suffix and version tags from model name."""
    base = name.split(':')[0]
    base = re.sub(r'-cuda.*', '', base)
    return base

# Use the actual resolved model name from connection cell
RAW_MODEL = model_name
ALT_MODEL = _strip_model_name(RAW_MODEL)

def _try_via_client(messages, prompt, model_id: str, max_tokens=220, temperature=0.2):
    """Try generating response using OpenAI client with multiple fallback routes."""
    attempts = []
    
    # 1. Try chat.completions endpoint (preferred for chat models)
    try:
        resp = client.chat.completions.create(
            model=model_id, 
            messages=messages, 
            max_tokens=max_tokens, 
            temperature=temperature
        )
        content = resp.choices[0].message.content
        attempts.append(('chat.completions', 200, (content or '')[:160]))
        if content and content.strip():
            return content, attempts
    except Exception as e:
        attempts.append(('chat.completions', None, str(e)[:160]))
    
    # 2. Try legacy completions endpoint
    try:
        comp = client.completions.create(
            model=model_id, 
            prompt=prompt, 
            max_tokens=max_tokens, 
            temperature=temperature
        )
        txt = comp.choices[0].text if comp.choices else ''
        attempts.append(('completions', 200, (txt or '')[:160]))
        if txt and txt.strip():
            return txt, attempts
    except Exception as e:
        attempts.append(('completions', None, str(e)[:160]))
    
    return None, attempts

def retrieve(query, k=3):
    """Retrieve top-k most similar documents using cosine similarity."""
    if embedder is None or doc_emb is None:
        raise RuntimeError("Embeddings not initialized.")
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_emb @ q_emb
    idxs = np.argsort(scores)[::-1][:k]
    return idxs

def answer(query, k=3, max_tokens=220, temperature=0.2, try_alternate=True):
    """
    Answer a query using RAG pipeline:
    1. Retrieve relevant documents using vector similarity
    2. Generate grounded response using Foundry Local model via OpenAI SDK
    
    Args:
        query: User question
        k: Number of documents to retrieve
        max_tokens: Maximum tokens for generation
        temperature: Sampling temperature
        try_alternate: Whether to try alternate model name on failure
    
    Returns:
        Dictionary with query, answer, docs, context, route, and tried attempts
    """
    if client is None:
        raise RuntimeError('Model client not initialized. Re-run connection cell after starting Foundry Local.')
    if embedder is None or doc_emb is None:
        raise RuntimeError('Embeddings not initialized.')
    
    # Retrieve relevant documents
    idxs = retrieve(query, k=k)
    context = '\n'.join(f'Doc {i}: {DOCS[i]}' for i in idxs)
    
    # Construct grounded generation prompt
    system_content = 'Use ONLY provided context. If insufficient, say "I\'m not sure."'
    user_content = f'Context:\n{context}\n\nQuestion: {query}'
    messages = [
        {'role': 'system', 'content': system_content},
        {'role': 'user', 'content': user_content}
    ]
    prompt = f'System: {system_content}\n{user_content}\nAnswer:'
    
    # Try generation with primary model
    tried = []
    ans, attempts = _try_via_client(messages, prompt, RAW_MODEL, max_tokens=max_tokens, temperature=temperature)
    tried.append({'model': RAW_MODEL, 'attempts': attempts})
    
    if ans and ans.strip():
        return {
            'query': query, 
            'answer': ans.strip(), 
            'docs': idxs.tolist(), 
            'context': context, 
            'route': 'chat-first', 
            'tried': tried
        }
    
    # Try alternate model name if available
    if try_alternate and ALT_MODEL != RAW_MODEL:
        ans2, attempts2 = _try_via_client(messages, prompt, ALT_MODEL, max_tokens=max_tokens, temperature=temperature)
        tried.append({'model': ALT_MODEL, 'attempts': attempts2})
        if ans2 and ans2.strip():
            return {
                'query': query, 
                'answer': ans2.strip(), 
                'docs': idxs.tolist(), 
                'context': context, 
                'route': 'chat-alt', 
                'tried': tried
            }
    
    # All routes failed
    return {
        'query': query, 
        'answer': 'I\'m not sure. (All SDK routes failed)', 
        'docs': idxs.tolist(), 
        'context': context, 
        'route': 'failed', 
        'tried': tried
    }

print('[INFO] SDK generation mode active.')
print(f'       RAW_MODEL = {RAW_MODEL}')
print(f'       ALT_MODEL = {ALT_MODEL}')

[INFO] SDK generation mode active.
       RAW_MODEL = deepseek-r1-distill-qwen-7b-cuda-gpu:0
       ALT_MODEL = deepseek-r1-distill-qwen-7b


In [15]:
# Self-test cell: validates connectivity, embeddings, and answer() basic functionality (SDK mode)
import math, pprint

def rag_self_test(sample_query: str = 'Why use RAG with local inference?', expect_docs: int = 3):
    report = {'base': BASE, 'raw_model': RAW_MODEL, 'alt_model': ALT_MODEL}
    if not BASE:
        report['error'] = 'BASE not resolved'
        return report
    if embedder is None or doc_emb is None:
        report['error'] = 'Embeddings not initialized'
        return report
    if getattr(doc_emb, 'shape', (0,))[0] != len(DOCS):
        report['warning_embeddings'] = f"doc_emb count {getattr(doc_emb,'shape',('?'))} mismatch DOCS {len(DOCS)}"
    try:
        idxs = retrieve(sample_query, k=expect_docs)
        report['retrieved_indices'] = idxs.tolist() if hasattr(idxs, 'tolist') else list(idxs)
    except Exception as e:
        report['error_retrieve'] = str(e)
        return report
    try:
        ans = answer(sample_query, k=expect_docs, max_tokens=80, temperature=0.2)
        report['route'] = ans.get('route')
        report['answer_preview'] = ans.get('answer','')[:160]
        if ans.get('route') == 'failed':
            report['warning_generation'] = 'All SDK routes failed for sample query'
    except Exception as e:
        report['error_generation'] = str(e)
    return report

pprint.pprint(rag_self_test())

{'alt_model': 'deepseek-r1-distill-qwen-7b',
 'answer_preview': 'Okay, so I need to figure out why someone would use '
                   'Retrieval Augmented Generation (RAG) with local inference. '
                   'Let me start by understanding each part of the qu',
 'base': 'http://127.0.0.1:59778',
 'raw_model': 'deepseek-r1-distill-qwen-7b-cuda-gpu:0',
 'retrieved_indices': [0, 3, 1],
 'route': 'chat-first'}


### ពន្យល់៖ សាកល្បងសំណួរជាច្រើន (Batch Query Smoke Test)
អនុវត្តសំណួរទម្រង់តំណាងច្រើនពីអ្នកប្រើតាមរយៈ `answer()` ដើម្បីផ្ទៀងផ្ទាត់:
- សន្ទស្សន៍ការទាញយកសមនឹងឯកសារគាំទ្រដែលមានសមរម្យ.
- ការបញ្ជូនជំនួសដំណើរការ ធ្វើការបាន (route value not 'failed').
- ចម្លើយគោរពការណែនាំផ្អែកលើមូលដ្ឋាន (គ្មានការស្រមៃ).
កត់ត្រាវត្ថុលទ្ធផលចុងក្រោយសម្រាប់ការត្រួតពិនិត្យបន្ទាន់.


In [16]:
# Quick test queries

queries = [

    "Why use RAG with local inference?",

    "What does vector similarity search do?",

    "Explain privacy benefits."

]



last_result = None

for q in queries:

    try:

        r = answer(q)

        last_result = r

        print(f"Q: {q}\nA: {r['answer']}\nDocs: {r['docs']}\n---")

    except Exception as e:

        print(f"Failed answering '{q}': {e}")



last_result

Q: Why use RAG with local inference?
A: Okay, so I need to figure out why someone would use Retrieval Augmented Generation (RAG) with local inference. Let me start by understanding each part of the question.

First, RAG. From the context given, Doc 1 says that RAG improves answer grounding by injecting relevant context. So RAG is a method that uses retrieval techniques to find the most relevant parts of a document or corpus to augment the generation process. This probably helps in making the generated answers more accurate because they're backed by real data.

Then, local inference. Doc 0 mentions that Foundry Local provides an OpenAI-compatible local inference endpoint. So local inference means running the model on the user's device rather than sending the request to a remote server. This is good for privacy and reducing latency, but it might have limitations in terms of model size or capabilities compared to cloud-based options.

Now, combining RAG with local inference. The context s

{'query': 'Explain privacy benefits.',
 'answer': 'Okay, so I need to explain the privacy benefits mentioned in the provided context. Let me look at the context again. The context includes three documents:\n\nDoc 2 says Edge AI reduces latency and preserves privacy via local execution.\nDoc 3 mentions Small Language Models can offer competitive quality with lower resource usage.\nDoc 1 states Retrieval Augmented Generation improves answer grounding by injecting relevant context.\n\nThe question is about explaining the privacy benefits. So, I should focus on the parts of the context that talk about privacy. \n\nLooking at Doc 2, it mentions Edge AI reduces latency and preserves privacy via local execution. That seems directly related to privacy. I think "local execution" means that the AI processes data on the device itself rather than sending it to a server. This could mean that data doesn\'t have to be transmitted, which might help protect user privacy because it avoids centralizing d

### ការពន្យល់: ការ​ហៅ​ងាយស្រួល​សម្រាប់​ចម្លើយ​តែមួយ

ការ​ហៅ​ចុងក្រោយ​ដែល​ឆាប់សម្រាប់​សំណួរ​តែមួយ ដើម្បីងាយស្រួល copy/paste សម្រាប់ប្រើឡើងវិញ ឬយោងក្រោយ។ បង្ហាញពីការប្រើ `answer()` ដែលផ្តល់លទ្ធផលដូចគ្នា (idempotent) បន្ទាប់ពីសំណួរត្រៀមមុននេះ។


In [17]:
result = answer('Why use RAG with local inference?')
result

{'query': 'Why use RAG with local inference?',
 'answer': "Okay, so I need to figure out why someone would use Retrieval Augmented Generation (RAG) with local inference. Let me start by understanding each part of the question.\n\nFirst, RAG. From the context given, Doc 1 says that RAG improves answer grounding by injecting relevant context. So RAG is a method that uses retrieval techniques to find the most relevant parts of a document or corpus to augment the generation process. This probably helps in making the generated answers more accurate because they're backed by real data.\n\nThen, local inference. Doc 0 mentions that Foundry Local provides an OpenAI-compatible local inference endpoint. So local inference means running the model on the user's device rather than sending the request to a remote server. This is good for privacy and reducing latency, but it might have limitations in terms of model size or capabilities compared to cloud-based options.\n\nNow, combining RAG with local

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារ​នេះ​ត្រូវបានបកប្រែ​ដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). បើទោះបី​យើងខិតខំស្វែងរកភាពត្រឹមត្រូវក៏ដោយ សូមជ្រាបថា ការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកជាប្រភពដែលមានអំណាច។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមផ្តល់អនុសាសន៍ឲ្យមានការបកប្រែដោយអ្នកជំនាញមនុស្សវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយ ដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
